# Session 21: GPU Computing in Python
## CPU vs GPU -- What's the Difference?

- **CPU**: A few powerful cores (72 on this node). Good at complex, sequential tasks.
- **GPU**: Thousands of small cores (16,896 on the H100). Good at doing the same operation on millions of data points at once.

Today we'll use **CuPy** (GPU-accelerated NumPy) to see the difference firsthand.

---
## Part 0: Check Your GPU

In [ ]:
import os
import time
import numpy as np

# Check GPU
gpu_info = os.popen('nvidia-smi').read()
print(gpu_info)

if 'NVIDIA' not in gpu_info:
    print("No GPU detected! Make sure you submitted to the gh-dev queue on Vista.")

In [ ]:
# Install CuPy if needed (GPU-accelerated NumPy)
try:
    import cupy as cp
    print(f"CuPy version: {cp.__version__}")
    print(f"GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
    print(f"GPU Memory: {cp.cuda.runtime.getDeviceProperties(0)['totalGlobalMem'] / 1e9:.0f} GB")
except ImportError:
    print("Installing CuPy... (this takes a minute)")
    os.system('pip install --user cupy-cuda12x 2>/dev/null')
    import cupy as cp
    print(f"CuPy installed! Version: {cp.__version__}")

---
## Part 1: Matrix Multiplication -- CPU vs GPU

Same operation, same data, different hardware. CuPy uses the exact same syntax as NumPy but runs on the GPU.

In [ ]:
sizes = [1000, 2000, 5000, 10000, 15000, 20000]
cpu_times = []
gpu_times = []

print(f"{'Size':>8} {'CPU (sec)':>12} {'GPU (sec)':>12} {'Speedup':>10}")
print("-" * 46)

for n in sizes:
    # CPU (NumPy)
    A_cpu = np.random.randn(n, n).astype(np.float32)
    B_cpu = np.random.randn(n, n).astype(np.float32)
    
    start = time.time()
    C_cpu = A_cpu @ B_cpu
    t_cpu = time.time() - start
    cpu_times.append(t_cpu)
    
    # GPU (CuPy)
    A_gpu = cp.asarray(A_cpu)
    B_gpu = cp.asarray(B_cpu)
    
    # Warm up GPU
    _ = A_gpu @ B_gpu
    cp.cuda.Stream.null.synchronize()
    
    start = time.time()
    C_gpu = A_gpu @ B_gpu
    cp.cuda.Stream.null.synchronize()  # Wait for GPU to finish
    t_gpu = time.time() - start
    gpu_times.append(t_gpu)
    
    speedup = t_cpu / t_gpu
    print(f"{n:>8} {t_cpu:>12.4f} {t_gpu:>12.4f} {speedup:>9.1f}x")
    
    # Free GPU memory
    del A_gpu, B_gpu, C_gpu
    cp.get_default_memory_pool().free_all_blocks()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time comparison
x = range(len(sizes))
width = 0.35
axes[0].bar([i - width/2 for i in x], cpu_times, width, label='CPU (NumPy)', color='maroon')
axes[0].bar([i + width/2 for i in x], gpu_times, width, label='GPU (CuPy)', color='#4a6fa5')
axes[0].set_xticks(x)
axes[0].set_xticklabels([str(s) for s in sizes])
axes[0].set_xlabel('Matrix Size', fontsize=12)
axes[0].set_ylabel('Time (seconds)', fontsize=12)
axes[0].set_title('CPU vs GPU: Matrix Multiplication', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Speedup
speedups = [c / g for c, g in zip(cpu_times, gpu_times)]
axes[1].plot(sizes, speedups, 'o-', color='maroon', linewidth=2, markersize=8)
axes[1].set_xlabel('Matrix Size', fontsize=12)
axes[1].set_ylabel('GPU Speedup (x times faster)', fontsize=12)
axes[1].set_title('GPU Speedup Over CPU', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=1, color='gray', linestyle='--', label='Break-even')
axes[1].legend()

plt.tight_layout()
plt.show()

### YOUR TURN

**1. What was the GPU speedup for the largest matrix size?**

_Your answer:_


**2. For small matrices (1000x1000), is the GPU always faster? Why or why not?**

_Your answer:_



---
## Part 2: Monte Carlo on GPU -- Millions of Random Numbers

Remember the Monte Carlo pi estimation from Tuesday? Let's run it on the GPU with 1 billion samples.

In [ ]:
N_SAMPLES = 1_000_000_000  # 1 billion

# CPU version
print(f"Monte Carlo pi estimation with {N_SAMPLES:,} samples")
print()

print("CPU (NumPy)...")
start = time.time()
x_cpu = np.random.random(N_SAMPLES).astype(np.float32)
y_cpu = np.random.random(N_SAMPLES).astype(np.float32)
inside_cpu = np.sum(x_cpu**2 + y_cpu**2 <= 1.0)
pi_cpu = 4.0 * inside_cpu / N_SAMPLES
t_cpu = time.time() - start
print(f"  Pi = {pi_cpu:.6f} | Time: {t_cpu:.2f}s")
del x_cpu, y_cpu  # Free RAM

# GPU version
print("\nGPU (CuPy)...")
start = time.time()
x_gpu = cp.random.random(N_SAMPLES, dtype=cp.float32)
y_gpu = cp.random.random(N_SAMPLES, dtype=cp.float32)
inside_gpu = cp.sum(x_gpu**2 + y_gpu**2 <= 1.0)
pi_gpu = 4.0 * float(inside_gpu) / N_SAMPLES
cp.cuda.Stream.null.synchronize()
t_gpu = time.time() - start
print(f"  Pi = {pi_gpu:.6f} | Time: {t_gpu:.2f}s")
del x_gpu, y_gpu
cp.get_default_memory_pool().free_all_blocks()

print(f"\nActual pi: {np.pi:.6f}")
print(f"GPU speedup: {t_cpu / t_gpu:.1f}x")
print(f"\nWe just generated and processed {N_SAMPLES * 2:,} random numbers.")

---
## Part 3: Image Processing at Scale

GPUs were originally built for graphics. Let's use one for batch image processing -- applying filters to hundreds of images simultaneously.

In [ ]:
# Simulate batch image processing: apply a blur filter to 500 images
N_IMAGES = 500
IMG_SIZE = 1024  # 1024x1024 pixels, 3 color channels

print(f"Processing {N_IMAGES} images ({IMG_SIZE}x{IMG_SIZE} RGB)")
print(f"Total data: {N_IMAGES * IMG_SIZE * IMG_SIZE * 3 * 4 / 1e9:.1f} GB (float32)")
print()

# Create a simple 5x5 blur kernel
kernel_size = 5
kernel = np.ones((kernel_size, kernel_size), dtype=np.float32) / (kernel_size**2)

def apply_blur_cpu(images, kernel):
    """Apply blur using convolution on CPU."""
    from scipy.signal import fftconvolve
    results = []
    for img in images:
        blurred = np.stack([
            fftconvolve(img[:, :, c], kernel, mode='same')
            for c in range(3)
        ], axis=-1)
        results.append(blurred)
    return results

# Generate random images (simulating real image data)
print("Generating random images...")
images_cpu = [np.random.rand(IMG_SIZE, IMG_SIZE, 3).astype(np.float32) for _ in range(20)]  # Use 20 for CPU

# CPU: process 20 images
print(f"CPU: processing 20 images...")
start = time.time()
_ = apply_blur_cpu(images_cpu, kernel)
t_cpu_per_image = (time.time() - start) / 20
t_cpu_estimated = t_cpu_per_image * N_IMAGES
print(f"  {t_cpu_per_image:.3f}s per image")
print(f"  Estimated time for {N_IMAGES} images: {t_cpu_estimated:.1f}s")

# GPU: process all 500 images using batch operations
print(f"\nGPU: processing {N_IMAGES} images in batch...")
from cupyx.scipy.signal import fftconvolve as gpu_fftconvolve

kernel_gpu = cp.asarray(kernel)
start = time.time()

# Process in batches of 50 to manage GPU memory
batch_size = 50
for i in range(0, N_IMAGES, batch_size):
    batch = cp.random.rand(batch_size, IMG_SIZE, IMG_SIZE, 3, dtype=cp.float32)
    for b in range(batch_size):
        for c in range(3):
            gpu_fftconvolve(batch[b, :, :, c], kernel_gpu, mode='same')
    del batch
    cp.get_default_memory_pool().free_all_blocks()

cp.cuda.Stream.null.synchronize()
t_gpu_total = time.time() - start
print(f"  Total time: {t_gpu_total:.1f}s")
print(f"  {t_gpu_total / N_IMAGES:.3f}s per image")

print(f"\nGPU speedup: {t_cpu_estimated / t_gpu_total:.1f}x")

### YOUR TURN

**1. How much total data (in GB) did we process across all 500 images?**

_Your answer:_


**2. What real-world applications need to process hundreds or thousands of images?** (Name at least 2)

_Your answer:_



---
## Part 4: When to Use CPU vs GPU

GPUs aren't always the answer. Here's when each makes sense:

| Use CPU When... | Use GPU When... |
|----------------|----------------|
| Small data (< few MB) | Large data (GB+) |
| Complex branching logic | Same operation on millions of elements |
| File I/O, string processing | Matrix math, image processing |
| Few tasks with many steps | Many tasks with few steps |
| Code needs to be simple | Performance is critical |

**The GPU wins when the problem is big enough to keep thousands of cores busy. For small problems, the overhead of moving data to the GPU makes it slower.**

---
## Exit Ticket

Answer these before you leave. Submit on Canvas.

**1. What GPU is on your Vista node? How much memory does it have?**

_Your answer:_


**2. For the 20,000x20,000 matrix multiplication, what was the GPU speedup over CPU?**

_Your answer:_


**3. True or False: A GPU is always faster than a CPU for any computation. Explain.**

_Your answer:_


**4. Give one example from your field of study where GPU computing could be useful.**

_Your answer:_

